In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# SOURCE
source_table = "ecommerce_analytics.bronze.sales"
df = spark.read.table(source_table)
df.display()


In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import from_json

df = spark.read.table("ecommerce_analytics.bronze.sales")

product_schema = StructType([
    StructField("curr", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", DoubleType()),
    StructField("qty", IntegerType()),
    StructField("unit", StringType())
])

sales_df = df.withColumn("product", from_json("product", product_schema))\
                .select("customer_id", "customer_name", "product_name", "order_date", "product_category", "product.*", "total_price")

sales_df.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales")

In [0]:
# PRODUCT JSON SCHEMA
product_schema = StructType([
    StructField("curr", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", DoubleType()),
    StructField("qty", IntegerType()),
    StructField("unit", StringType())
])

########################################
# TRANSFORMATION
########################################

# 1. Parse JSON
df = df.withColumn("product_json", from_json(col("product"), product_schema))

# 2. Flatten product fields
df = df.withColumn("product_id", col("product_json.id")) \
       .withColumn("product_name_clean", col("product_json.name")) \
       .withColumn("price", col("product_json.price")) \
       .withColumn("quantity", col("product_json.qty")) \
       .withColumn("currency", col("product_json.curr")) \
       .withColumn("unit", col("product_json.unit"))

# 3. Clean customer name
df = df.withColumn("name_parts", split(col("customer_name"), ",")) \
       .withColumn("first_name", expr("element_at(name_parts, 2)")) \
       .withColumn("last_name", expr("element_at(name_parts, 1)"))

# 4. Type casting
df = df.withColumn("order_date", to_date(col("order_date")))

# 5. Derived columns
df = df.withColumn("calculated_total", col("price") * col("quantity")) \
       .withColumn("is_price_valid", col("calculated_total") == col("total_price")) \
       .withColumn("order_month", expr("date_format(order_date, 'yyyy-MM')")) \
       .withColumn("order_year", expr("year(order_date)"))

# 6. Metadata
df = df.withColumn("silver_load_ts", current_timestamp())

# 7. Drop unnecessary columns
df = df.drop(
    "product",
    "product_json",
    "file_path",
    "name_parts",
    "product_name"   # unreliable
)

# 8. Deduplication
df = df.dropDuplicates(["customer_id", "product_id", "order_date"])
